# BERT (DistilBERT) Sentiment Classifier

Fine-tuning DistilBERT for IMDB sentiment classification using HuggingFace Transformers.

## Imports

In [1]:
import pandas as pd
import torch
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset

c:\Users\misag\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
SAVE_DIR = "models/bert_sentiment"
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 2e-5

# For quick verification, use a subset. Set to None for full dataset.
N_SAMPLES = None  # Set to None for full training

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Load Tokenizer and Data

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading data...")
train_df = pd.read_csv("data/train.csv").dropna(subset=['review'])
val_df = pd.read_csv("data/val.csv").dropna(subset=['review'])

if N_SAMPLES:
    train_df = train_df.sample(N_SAMPLES, random_state=42)
    val_df = val_df.sample(N_SAMPLES // 2, random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")

Loading data...
Train size: 35000
Val size: 7500


## Prepare HuggingFace Datasets

In [4]:
def tokenize_function(examples):
    return tokenizer(examples["review"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

# Convert to HuggingFace Datasets
train_hf = Dataset.from_pandas(train_df[['review', 'sentiment']])
val_hf = Dataset.from_pandas(val_df[['review', 'sentiment']])

# Encode labels
train_hf = train_hf.map(lambda x: {"label": 1 if x["sentiment"] == 'positive' else 0})
val_hf = val_hf.map(lambda x: {"label": 1 if x["sentiment"] == 'positive' else 0})

# Tokenize
print("Tokenizing datasets...")
train_dataset = train_hf.map(tokenize_function, batched=True)
val_dataset = val_hf.map(tokenize_function, batched=True)

Map: 100%|██████████| 7500/7500 [00:00<00:00, 63798.94 examples/s]


Tokenizing datasets...


Map: 100%|██████████| 7500/7500 [00:01<00:00, 6066.09 examples/s]


## Define Metrics

In [5]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

## Initialize Model and Trainer

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 12435.30it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## Train the Model

In [7]:
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.234643,0.255271,0.905733,0.907376,0.891836,0.923467
2,0.178479,0.276762,0.914667,0.916536,0.896886,0.937067
3,0.081882,0.341086,0.916267,0.916622,0.912745,0.920533


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.68it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=6564, training_loss=0.18079032329870826, metrics={'train_runtime': 591.2606, 'train_samples_per_second': 177.587, 'train_steps_per_second': 11.102, 'total_flos': 6954538429440000.0, 'train_loss': 0.18079032329870826, 'epoch': 3.0})

## Save the Model

In [8]:
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.93it/s]

Model saved to models/bert_sentiment


## Quick Prediction Test

In [9]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_label_id = torch.argmax(probs, dim=1).item()
    sentiment = "positive" if pred_label_id == 1 else "negative"
    confidence = probs[0][pred_label_id].item()
    return sentiment, confidence

text = "This movie was absolutely wonderful!"
sentiment, confidence = predict(text)
print(f"Text: {text}")
print(f"Prediction: {sentiment} (confidence: {confidence:.4f})")

Text: This movie was absolutely wonderful!
Prediction: positive (confidence: 0.9949)
